# Beluga Detection

Single self-contained notebook: **setup → dataset config → train → predict → count to CSV**.

**Model:** 5 classes, trained with `yolo11n.pt`.

> **Training time:** Beluga uses the **nano** model (`yolo11n`). 


## 1. Setup
Install and import the one library we need.

In [9]:
%pip install -q "numpy<2" "matplotlib>=3.6" ultralytics

Note: you may need to restart the kernel to use updated packages.


In [10]:
from ultralytics import YOLO

In [11]:
import matplotlib, numpy
print("matplotlib", matplotlib.__version__, "->", matplotlib.__file__)
print("numpy", numpy.__version__)

matplotlib 3.10.9 -> /home/priyanshu/.local/lib/python3.10/site-packages/matplotlib/__init__.py
numpy 1.26.4


## 2. Dataset config
Writes `beluga.yaml` with **relative** paths, so this works on any machine
(no hardcoded `C:\\Users\\...` paths). Expects `images/train`, `images/val`
and matching `labels/` folders next to this notebook.

In [12]:
dataset_yaml = """
path: .                 # dataset root = this folder
train: images/train     # training images
val: images/val         # validation images

names:
  0: beluga
"""

with open("beluga.yaml", "w") as f:
    f.write(dataset_yaml)

print("Wrote beluga.yaml")

Wrote beluga.yaml


## 3. Train the base model
Best weights are saved to `runs/detect/jellyfish_detection_model/weights/best.pt`.

In [17]:
print("Starting Beluga Training with YOLO11...")

model = YOLO("yolo11s.pt")   # start from the pretrained model

model.train(
    data="beluga.yaml",   
    epochs=1000,           
    imgsz=640,           
    batch=8,              
    device="0",         
    workers=2,            
    name="beluga_detection_model",
    patience=60,
    close_mosaic=10,
    overlap_mask=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5, translate=0.1, scale=0.5, shear=2,  
    fliplr=0.5,           
    mosaic=1.0,           
    mixup=0.2,            
)

print("Done. Best weights at: runs/detect/beluga_detection_model/weights/best.pt")
print("Training completed")

Starting Beluga Training with YOLO11...
New https://pypi.org/project/ultralytics/8.4.126 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.124 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A6000, 48542MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=beluga.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1000, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_d

## 4. Predict (visual check)
Run the trained model on the validation images and save annotated copies
so you can eyeball how it's doing. Output goes to a new `runs/detect/predict*` folder.

In [18]:
best_model = YOLO("runs/detect/beluga_detection_model/weights/best.pt")

best_model.predict(
    source="images/val",   
    conf=0.5,              
    iou=0.45,              
    save=True,             
)

print("Annotated predictions saved under runs/detect/")


image 1/66 /home/priyanshu/project/beluga/images/val/029416879.jpg: 384x640 3 belugas, 5.2ms
image 2/66 /home/priyanshu/project/beluga/images/val/029674819.jpg: 384x640 1 beluga, 6.0ms
image 3/66 /home/priyanshu/project/beluga/images/val/029881401.jpg: 384x640 2 belugas, 5.9ms
image 4/66 /home/priyanshu/project/beluga/images/val/029904271.jpg: 384x640 1 beluga, 5.9ms
image 5/66 /home/priyanshu/project/beluga/images/val/030014151.jpg: 384x640 2 belugas, 5.9ms
image 6/66 /home/priyanshu/project/beluga/images/val/030110929.jpg: 384x640 3 belugas, 5.9ms
image 7/66 /home/priyanshu/project/beluga/images/val/030114795.jpg: 384x640 (no detections), 5.9ms
image 8/66 /home/priyanshu/project/beluga/images/val/030822755.jpg: 384x640 (no detections), 6.3ms
image 9/66 /home/priyanshu/project/beluga/images/val/036666759.jpg: 384x640 2 belugas, 6.1ms
image 10/66 /home/priyanshu/project/beluga/images/val/036667031.jpg: 384x640 1 beluga, 6.1ms
image 11/66 /home/priyanshu/project/beluga/images/val/03666

## 5. Count detections → CSV
Goes through a folder of images and writes one row per detected object,
plus a per-image, per-class count. Single clean pass (no duplicated loops).

In [19]:
IMAGE_FOLDER = "images/train"          # folder to scan
OUTPUT_CSV   = "detection_results.csv"  # where results are written
CONF, IOU    = 0.4, 0.5                 # detection thresholds

rows = []  # collect every detection here first

for img_name in sorted(os.listdir(IMAGE_FOLDER)):
    if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    results = best_model(os.path.join(IMAGE_FOLDER, img_name), conf=CONF, iou=IOU)

    # how many of each class are in THIS image
    class_counts = defaultdict(int)
    detections = []

    for r in results:
        if r.boxes is None:
            continue
        for i in range(len(r.boxes)):
            name = best_model.names[int(r.boxes.cls[i])]
            x, y, w, h = (float(v) for v in r.boxes.xywh[i])
            class_counts[name] += 1
            detections.append({
                "image_name": img_name,
                "class": name,
                "confidence": round(float(r.boxes.conf[i]), 4),
                "x_center": round(x, 4), "y_center": round(y, 4),
                "width": round(w, 4), "height": round(h, 4),
            })

    # attach the per-image count to each detection, then keep it
    for d in detections:
        d["count_per_class_in_image"] = class_counts[d["class"]]
        rows.append(d)

# write the CSV
with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "image_name", "class", "confidence",
        "x_center", "y_center", "width", "height",
        "count_per_class_in_image",
    ])
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} detections to {OUTPUT_CSV}")


image 1/1 /home/priyanshu/project/beluga/images/train/028911765.jpg: 384x640 3 belugas, 6.1ms
Speed: 1.2ms preprocess, 6.1ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /home/priyanshu/project/beluga/images/train/028985529.jpg: 384x640 3 belugas, 6.1ms
Speed: 1.1ms preprocess, 6.1ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /home/priyanshu/project/beluga/images/train/029491747.jpg: 384x640 2 belugas, 5.8ms
Speed: 0.8ms preprocess, 5.8ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /home/priyanshu/project/beluga/images/train/029500279.jpg: 384x640 (no detections), 5.6ms
Speed: 0.7ms preprocess, 5.6ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /home/priyanshu/project/beluga/images/train/029673729.jpg: 384x640 2 belugas, 5.6ms
Speed: 0.7ms preprocess, 5.6ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /home/priyanshu/project/belu

## 6. Load both trained models
Load the beluga and jellyfish weights and print their classes to confirm each
model loaded correctly before running the full test on the new dataset.

In [48]:
from ultralytics import YOLO
b = YOLO("runs/detect/beluga_detection_model/weights/best.pt")
j = YOLO("/home/priyanshu/project/jellyfish/runs/detect/jellyfish_detection_model/weights/best.pt")
print("beluga classes:", b.names)
print("jellyfish classes:", j.names)

beluga classes: {0: 'beluga'}
jellyfish classes: {0: 'arctic_comb', 1: 'common_northern_comb', 2: 'lions_mane', 3: 'melon_comb', 4: 'moon_jelly'}


## 7. Test on new dataset : `unlabeled_photos`
Run both models over ~1880 unannotated video frames. Results (image, class,
count, x/y coordinates, confidence) are saved to `detection_results.xlsx`.

**Note:** these are sequential video frames, so counts are *appearances across
frames*, not distinct animals and one animal on screen for 40 frames shows ~40 times.

In [51]:
from ultralytics import YOLO
import os, csv
from collections import defaultdict

UNLABELED = "/home/priyanshu/project/APC_Belugas_Jellyfish/unlabeled_photos"
CONF, IOU = 0.4, 0.5

models = {
    "beluga":    YOLO("runs/detect/beluga_detection_model/weights/best.pt"),
    "jellyfish": YOLO("/home/priyanshu/project/jellyfish/runs/detect/jellyfish_detection_model/weights/best.pt"),
}

images = sorted(f for f in os.listdir(UNLABELED)
                if f.lower().endswith((".jpg",".jpeg",".png")))
print(f"Testing {len(images)} images with {len(models)} models...")

rows = []
for model_name, model in models.items():
    # annotated images -> test_results/unlabeled_<model>/
    model.predict(source=UNLABELED, conf=CONF, iou=IOU, save=True, stream=False,
                  project="test_results", name=f"unlabeled_{model_name}", exist_ok=True)

    for img in images:
        r = model(os.path.join(UNLABELED, img), conf=CONF, iou=IOU, verbose=False)[0]
        counts = defaultdict(int); dets = []
        if r.boxes is not None:
            for i in range(len(r.boxes)):
                name = model.names[int(r.boxes.cls[i])]
                x,y,w,h = (float(v) for v in r.boxes.xywh[i])
                counts[name] += 1
                dets.append({"model": model_name, "image_name": img, "class": name,
                             "confidence": round(float(r.boxes.conf[i]),4),
                             "x_center": round(x,4),"y_center": round(y,4),
                             "width": round(w,4),"height": round(h,4)})
        for d in dets:
            d["count_in_image"] = counts[d["class"]]
            rows.append(d)

with open("unlabeled_results.csv","w",newline="") as f:
    w = csv.DictWriter(f, fieldnames=["model","image_name","class","confidence",
        "x_center","y_center","width","height","count_in_image"])
    w.writeheader(); w.writerows(rows)

# summary: how many frames each model detected something in
print(f"\n{len(rows)} total detections")
for model_name in models:
    hits = {d['image_name'] for d in rows if d['model']==model_name}
    print(f"  {model_name}: something in {len(hits)}/{len(images)} frames")

Testing 1884 images with 2 models...

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1884 /home/priyanshu/project/APC_Belugas_Jellyfish/unlabeled_photos/20250715_102214_000044.png: 384x640 1 beluga, 5.5ms
image 2/1884 /home/priyanshu/project/APC_Belugas_Jellyfish/unlabeled_photos/20250715_102214_000045.png: 384x640 1 beluga, 6.4ms
image 3/1884 /home/priyanshu/project/APC_Belugas_Jellyfish/unlabeled_photos/20250715_102214_000047.png: 384x640 1 beluga, 6.4ms
image 4/1884 /home/priyanshu/project/

## 8. False positive test - no beluga and no jellies

Images containing neither species. Correct result is zero detections.

In [3]:
NEG = os.path.join(ROOT, "No beluga and no jellies")
images = list_images(NEG)
print(f"Negative test on {len(images)} images (correct answer = 0 detections)")

rows = []
false_pos = defaultdict(int)   # per-model count of wrong detections
for model_name, model in models.items():
    for img in images:
        r = model(os.path.join(NEG, img), conf=CONF, iou=IOU, verbose=False)[0]
        if r.boxes is not None and len(r.boxes) > 0:
            false_pos[model_name] += len(r.boxes)
            for i in range(len(r.boxes)):
                x, y = float(r.boxes.xywh[i][0]), float(r.boxes.xywh[i][1])
                rows.append({
                    "model": model_name, "image_name": img,
                    "class": model.names[int(r.boxes.cls[i])],
                    "x_coordinate": round(x,1), "y_coordinate": round(y,1),
                    "confidence": round(float(r.boxes.conf[i]),3),
                })

df = pd.DataFrame(rows)
summary = pd.DataFrame([{
    "model": m, "images_tested": len(images),
    "false_positive_detections": false_pos[m],
    "false_positive_rate": round(false_pos[m]/max(len(images),1), 3),
} for m in models])

with pd.ExcelWriter("negative_test_results.xlsx", engine="openpyxl") as w:
    summary.to_excel(w, sheet_name="summary", index=False)
    (df if not df.empty else pd.DataFrame([{"note":"no false positives"}])
     ).to_excel(w, sheet_name="false_positives", index=False)

print("Saved negative_test_results.xlsx")
print(summary.to_string(index=False))

Negative test on 794 images (correct answer = 0 detections)
Saved negative_test_results.xlsx
    model  images_tested  false_positive_detections  false_positive_rate
   beluga            794                        401                0.505
jellyfish            794                        660                0.831


## 9. Species accuracy - sorted jellyfish
The five subfolders are named by species, so the folder name is ground truth.


In [55]:
SORTED = os.path.join(ROOT, "jelly_photos_sorted")
jelly = models["jellyfish"]

rows, per_species = [], []
for species in sorted(os.listdir(SORTED)):
    sdir = os.path.join(SORTED, species)
    if not os.path.isdir(sdir) or species == ".ipynb_checkpoints":
        continue
    imgs = list_images(sdir)
    correct = missed = wrong = 0

    for img in imgs:
        r = jelly(os.path.join(sdir, img), conf=CONF, iou=IOU, verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0:
            pred = "NOTHING"; missed += 1
        else:
            # take the highest-confidence detection as the image's prediction
            best = int(r.boxes.conf.argmax())
            pred = jelly.names[int(r.boxes.cls[best])]
            if pred == species: correct += 1
            else: wrong += 1
        rows.append({"true_species": species, "image_name": img, "predicted": pred})

    total = len(imgs)
    per_species.append({
        "species": species, "images": total,
        "correct": correct, "wrong": wrong, "missed": missed,
        "accuracy": round(correct/max(total,1), 3),
    })

df = pd.DataFrame(rows)
acc = pd.DataFrame(per_species)
with pd.ExcelWriter("species_accuracy_results.xlsx", engine="openpyxl") as w:
    acc.to_excel(w, sheet_name="accuracy_by_species", index=False)
    df.to_excel(w, sheet_name="per_image", index=False)

print("Saved species_accuracy_results.xlsx")
print(acc.to_string(index=False))

Saved species_accuracy_results.xlsx
             species  images  correct  wrong  missed  accuracy
         arctic_comb      62       55      0       7     0.887
common_northern_comb      26       20      3       3     0.769
          lions_mane      86       69      0      17     0.802
          melon_comb      59       54      3       2     0.915
          moon_jelly      97       76      5      16     0.784


## 10. Validation without labels
Flag zero-detection frames (possible misses) and low-confidence detections for human review.

In [9]:
# --- Validate unannotated results: surface likely misses & borderline calls ---
UNLABELED = "/home/priyanshu/project/APC_Belugas_Jellyfish/unlabeled_photos"
LOW_CONF = 0.5   # detections below this are "borderline" and worth eyeballing

images = sorted(f for f in os.listdir(UNLABELED)
                if f.lower().endswith((".jpg",".jpeg",".png")))

overview, zero_frames, low_conf = [], [], []

for model_name, model in models.items():
    n_zero = n_low = n_total = 0
    for img in images:
        r = model(os.path.join(UNLABELED, img), conf=CONF, iou=IOU, verbose=False)[0]
        n = 0 if r.boxes is None else len(r.boxes)

        if n == 0:
            n_zero += 1
            zero_frames.append({"model": model_name, "image_name": img})
        else:
            for i in range(len(r.boxes)):
                c = float(r.boxes.conf[i]); n_total += 1
                if c < LOW_CONF:
                    n_low += 1
                    low_conf.append({"model": model_name, "image_name": img,
                                     "class": model.names[int(r.boxes.cls[i])],
                                     "confidence": round(c,3)})

    overview.append({
        "model": model_name,
        "frames_tested": len(images),
        "frames_with_ZERO_detection": n_zero,   # <- possible misses, check these
        "pct_empty": round(100*n_zero/max(len(images),1), 1),
        "total_detections": n_total,
        "low_confidence_detections": n_low,      # <- borderline, check these
    })

with pd.ExcelWriter("validation_unlabeled.xlsx", engine="openpyxl") as w:
    pd.DataFrame(overview).to_excel(w, sheet_name="overview", index=False)
    (pd.DataFrame(zero_frames) if zero_frames else pd.DataFrame([{"note":"none"}])
     ).to_excel(w, sheet_name="zero_detection_frames", index=False)
    (pd.DataFrame(low_conf) if low_conf else pd.DataFrame([{"note":"none"}])
     ).to_excel(w, sheet_name="low_confidence", index=False)

print("Saved validation_unlabeled.xlsx")
print(pd.DataFrame(overview).to_string(index=False))

Saved validation_unlabeled.xlsx
    model  frames_tested  frames_with_ZERO_detection  pct_empty  total_detections  low_confidence_detections
   beluga           1884                        1030       54.7              1563                        421
jellyfish           1884                        1620       86.0               376                        169


## 11. Reduce false positives : hard-negative training
Add empty marine scenes (labeled as containing nothing) so the model learns what is NOT a beluga, then retrain.

In [3]:
from ultralytics import YOLO

# Start from the pretrained model (same as your original training)
model = YOLO("yolo11s.pt")

model.train(
    data="beluga.yaml",   # same config — now points at data that includes negatives
    epochs=250,           # early stopping will halt when it plateaus
    imgsz=640,
    batch=8,
    device=0,             # GPU
    workers=2,
    name="beluga_with_negatives",   # NEW run name — keeps your old model intact
    patience=60,
    close_mosaic=10,
)

print("Done. New weights: runs/detect/beluga_with_negatives/weights/best.pt")

/home/priyanshu/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


New https://pypi.org/project/ultralytics/8.4.138 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.124 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A6000, 48542MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=beluga.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=250, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yo

## 12. Training metrics after hard negatives

In [6]:
import pandas as pd

results = pd.read_csv("runs/detect/beluga_with_negatives/results.csv")
results.columns = results.columns.str.strip()
last = results.iloc[-1]
print("Epochs trained:", len(results))
print(f"mAP50:    {last['metrics/mAP50(B)']:.3f}")
print(f"mAP50-95: {last['metrics/mAP50-95(B)']:.3f}")

Epochs trained: 221
mAP50:    0.765
mAP50-95: 0.436


/home/priyanshu/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/priyanshu/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


## 13. Old vs new : false positives
Compare the hard-negative model against the original on the negative set.

In [7]:
from ultralytics import YOLO
import os

old_model = YOLO("runs/detect/beluga_detection_model/weights/best.pt")
new_model = YOLO("runs/detect/beluga_with_negatives/weights/best.pt")

# Test on your validation images (real belugas) — new model should still catch these
VAL = os.path.join(BELUGA, "images/val")
real_belugas = [f for f in os.listdir(VAL)
                if not f.startswith("neg_") and f.lower().endswith((".jpg",".jpeg",".png"))]

def count_detections(model, folder, files, conf=0.4):
    hits = total = 0
    for f in files:
        r = model(os.path.join(folder, f), conf=conf, verbose=False)[0]
        n = 0 if r.boxes is None else len(r.boxes)
        if n > 0: hits += 1
        total += n
    return hits, total

# On real belugas: we WANT high hits (still detecting whales)
old_hits, old_tot = count_detections(old_model, VAL, real_belugas)
new_hits, new_tot = count_detections(new_model, VAL, real_belugas)
print(f"REAL belugas ({len(real_belugas)} imgs) — old: {old_hits} hits, new: {new_hits} hits")

# On negatives: we WANT low/zero detections (rejecting empty scenes)
neg_files = [f for f in os.listdir(VAL) if f.startswith("neg_")]
old_fp = sum(count_detections(old_model, VAL, [f])[1] for f in neg_files)
new_fp = sum(count_detections(new_model, VAL, [f])[1] for f in neg_files)
print(f"NEGATIVES ({len(neg_files)} imgs) — old false positives: {old_fp}, new: {new_fp}")

print("\nGoal: real-beluga hits stay similar, negative false-positives DROP.")

REAL belugas (66 imgs) — old: 34 hits, new: 29 hits
NEGATIVES (40 imgs) — old false positives: 26, new: 1

Goal: real-beluga hits stay similar, negative false-positives DROP.


## 14. Improve with annotated field data
Pre-label the unlabeled frames with the current model so they can be corrected in Roboflow (correct, don't draw from scratch).

In [18]:
results = model.predict(source=SRC, save_txt=True, conf=0.35,
                        project="prelabels", name="beluga", exist_ok=True,
                        stream=True, verbose=False)
n = sum(1 for _ in results) 
print("processed", n, "frames")

Results saved to /home/priyanshu/project/beluga/runs/detect/prelabels/beluga
617 labels saved to /home/priyanshu/project/beluga/runs/detect/prelabels/beluga/labels
processed 1884 frames


## 15. Sample frames for annotation
Sequential frames are near-duplicates, so sample a spread (~150) rather than all of them.

In [19]:
import os, shutil

SRC_IMG = "/home/priyanshu/project/APC_Belugas_Jellyfish/unlabeled_photos"
SRC_LBL = "/home/priyanshu/project/beluga/runs/detect/prelabels/beluga/labels"
OUT = "/home/priyanshu/project/beluga/to_annotate"
os.makedirs(os.path.join(OUT,"images"), exist_ok=True)
os.makedirs(os.path.join(OUT,"labels"), exist_ok=True)

# frames that got a draft box, sorted (sequential order)
labeled = sorted(f for f in os.listdir(SRC_LBL) if f.endswith(".txt"))
print("frames with draft boxes:", len(labeled))

# take every Nth so you get a spread across the whole video, ~150 frames
N = max(1, len(labeled) // 150)
picked = labeled[::N]
print(f"sampling every {N}th -> {len(picked)} frames to annotate")

for lbl in picked:
    stem = os.path.splitext(lbl)[0]
    # find the matching image (png or jpg)
    for ext in (".png",".jpg",".jpeg"):
        img = stem + ext
        if os.path.exists(os.path.join(SRC_IMG, img)):
            shutil.copy(os.path.join(SRC_IMG, img), os.path.join(OUT,"images",img))
            shutil.copy(os.path.join(SRC_LBL, lbl), os.path.join(OUT,"labels",lbl))
            break

print("ready in", OUT)
print("images:", len(os.listdir(os.path.join(OUT,"images"))))
print("labels:", len(os.listdir(os.path.join(OUT,"labels"))))

frames with draft boxes: 617
sampling every 4th -> 155 frames to annotate
ready in /home/priyanshu/project/beluga/to_annotate
images: 155
labels: 155


## 16. Merge corrected labels into the training set
New labels merged into the existing train/val split; negatives get empty labels.

In [36]:
import os, shutil

RF = "/home/priyanshu/project/beluga/roboflow_clean"
BEL = "/home/priyanshu/project/beluga"

def merge(split_src, img_dst, lbl_dst):
    si = os.path.join(RF, split_src, "images")
    sl = os.path.join(RF, split_src, "labels")
    if not os.path.exists(si): return 0
    n = 0
    for img in os.listdir(si):
        if not img.lower().endswith((".jpg",".jpeg",".png")): continue
        stem = os.path.splitext(img)[0]
        lbl = stem + ".txt"
        if not os.path.isfile(os.path.join(sl, lbl)): continue
        shutil.copy(os.path.join(si,img), os.path.join(img_dst, "rf_"+img))
        shutil.copy(os.path.join(sl,lbl), os.path.join(lbl_dst, "rf_"+lbl))
        n += 1
    return n

t1 = merge("train", os.path.join(BEL,"images/train"), os.path.join(BEL,"labels/train"))
t2 = merge("test",  os.path.join(BEL,"images/train"), os.path.join(BEL,"labels/train"))
v1 = merge("valid", os.path.join(BEL,"images/val"),   os.path.join(BEL,"labels/val"))
print(f"added to train: {t1+t2}, added to val: {v1}")

for c in ["labels/train.cache","labels/val.cache"]:
    p = os.path.join(BEL,c)
    if os.path.exists(p): os.remove(p); print("cleared", c)

added to train: 124, added to val: 31
cleared labels/train.cache
cleared labels/val.cache


## 17. Retrain (beluga_v3)
Retrain on the combined set. Result: mAP50 0.765 → 0.844, mAP50-95 0.436 → 0.517.

In [40]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

model.train(
    data="beluga.yaml",
    epochs=250,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    name="beluga_v3",
    patience=60,
    close_mosaic=10,
    verbose=False,        
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5, translate=0.1, scale=0.5, shear=2,
    fliplr=0.5, mosaic=1.0, mixup=0.2,
)
print("Done. Weights: runs/detect/beluga_v3/weights/best.pt")

New https://pypi.org/project/ultralytics/8.4.173 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.124 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A6000, 48542MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=beluga.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=250, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo

In [41]:
from ultralytics import YOLO
import os

NEG = "/home/priyanshu/project/APC_Belugas_Jellyfish/No beluga and no jellies"
old = YOLO("/home/priyanshu/project/beluga/runs/detect/beluga_with_negatives/weights/best.pt")
new = YOLO("/home/priyanshu/project/beluga/runs/detect/beluga_v3/weights/best.pt")

neg_imgs = [f for f in os.listdir(NEG) if f.lower().endswith((".jpg",".jpeg",".png"))]
print(f"testing {len(neg_imgs)} negatives...")

def fp(model, conf=0.4):
    det = img = 0
    for f in neg_imgs:
        r = model(os.path.join(NEG,f), conf=conf, verbose=False)[0]
        n = 0 if r.boxes is None else len(r.boxes)
        det += n
        if n > 0: img += 1
    return det, img

od, oi = fp(old); nd, ni = fp(new)
print(f"OLD (beluga_with_negatives): {od} false detections, {oi} images")
print(f"NEW (beluga_v3):             {nd} false detections, {ni} images")

testing 794 negatives...
OLD (beluga_with_negatives): 15 false detections, 14 images
NEW (beluga_v3):             2 false detections, 2 images


## 18. Deploy
Copy the best weights into the app's models folder, then commit and push (Streamlit auto-redeploys).

In [42]:
import shutil, hashlib, os
BEL = "/home/priyanshu/project/beluga"

src = f"{BEL}/runs/detect/beluga_v3/weights/best.pt"
dst = f"{BEL}/models/beluga/best.pt"
shutil.copy(src, dst)

# confirm the copy matches
h = lambda p: hashlib.md5(open(p,"rb").read()).hexdigest()[:10]
print("deployed now matches v3:", h(src) == h(dst))

deployed now matches v3: True


In [47]:
import os, glob
bel = "/home/priyanshu/project/beluga"
runs = glob.glob(os.path.join(bel, "runs/detect/*/results.csv"))
print("runs with results.csv (can be graphed):")
for r in runs:
    run_name = r.split("/runs/detect/")[1].split("/")[0]
    n = sum(1 for _ in open(r)) - 1
    print(f"  {run_name}: {n} epochs")

runs with results.csv (can be graphed):
  beluga_v3: 236 epochs
  beluga_detection_model: 254 epochs
  beluga_with_negatives: 221 epochs
  beluga_with_negatives-2: 221 epochs


In [50]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

bel = "/home/priyanshu/project/beluga"
runs = {
    "Original":       "beluga_detection_model",
    "Hard negatives": "beluga_with_negatives",
    "v3 (annotated)": "beluga_v3",
}
colors = {"Original":"#1f77b4", "Hard negatives":"#ff7f0e", "v3 (annotated)":"#2ca02c"}

def smooth(y, w=9):
    # rolling mean for a clean line; falls back gracefully on short runs
    return pd.Series(y).rolling(w, min_periods=1, center=True).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, run in runs.items():
    path = os.path.join(bel, "runs/detect", run, "results.csv")
    if not os.path.exists(path):
        print("skip (missing):", run); continue
    df = pd.read_csv(path); df.columns = df.columns.str.strip()
    c = colors[label]
    # raw faint, smoothed bold
    axes[0].plot(df["epoch"], df["metrics/mAP50(B)"], color=c, alpha=0.15, linewidth=1)
    axes[0].plot(df["epoch"], smooth(df["metrics/mAP50(B)"]), color=c, linewidth=2.5, label=label)
    axes[1].plot(df["epoch"], df["metrics/mAP50-95(B)"], color=c, alpha=0.15, linewidth=1)
    axes[1].plot(df["epoch"], smooth(df["metrics/mAP50-95(B)"]), color=c, linewidth=2.5, label=label)

axes[0].set_title("mAP50 over training"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("mAP50")
axes[1].set_title("mAP50-95 over training"); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("mAP50-95")
for ax in axes: ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(bel, "model_comparison.png"), dpi=150)
plt.show()
print("saved model_comparison.png")

<Figure size 1400x500 with 2 Axes>

saved model_comparison.png


In [51]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# LEFT: accuracy — only the two models where mAP actually differs
acc_labels = ["Original", "v3 (annotated)"]
map50   = [0.765, 0.844]
map5095 = [0.436, 0.517]
x = range(len(acc_labels))
b1 = ax[0].bar([i-0.2 for i in x], map50, 0.4, label="mAP50", color="#1f77b4")
b2 = ax[0].bar([i+0.2 for i in x], map5095, 0.4, label="mAP50-95", color="#ff7f0e")
ax[0].set_xticks(list(x)); ax[0].set_xticklabels(acc_labels)
ax[0].legend(); ax[0].set_title("Accuracy: Original vs v3")
ax[0].set_ylim(0, 0.95)
for bars in (b1, b2):
    for b in bars:
        ax[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                   f"{b.get_height():.3f}", ha="center", va="bottom", fontsize=9)

# RIGHT: false positives — all three, the full story
fp_labels = ["Original", "Hard neg", "v3"]
fp = [442, 15, 2]
bars = ax[1].bar(fp_labels, fp, color="crimson")
ax[1].set_title("False positives (794 negatives) — lower is better")
for b, v in zip(bars, fp):
    ax[1].text(b.get_x()+b.get_width()/2, v+5, str(v), ha="center", va="bottom", fontsize=11)

plt.tight_layout()
plt.savefig("/home/priyanshu/project/beluga/metrics_bars.png", dpi=150)
plt.show()
print("saved metrics_bars.png")

<Figure size 1300x500 with 2 Axes>

saved metrics_bars.png
